# Steerling-8B Instruct: Concept Amplify

This notebook generates twice on the same neutral prompt, a normal and a steered run

**Requirements:** an interpretable Steerling checkpoint and a GPU with >= 18 GB VRAM.

## 1. Load the model

Amplify needs the **interpretable** model, which exposes the concept heads the injection is built from. We load via HuggingFace `AutoModel` and wrap it in `SteerlingGenerator`.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
from steerling import SteerlingGenerator, GenerationConfig, SteeringConfig

model = AutoModel.from_pretrained("guidelabs/steerling-8b-instruct", trust_remote_code=True, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained("guidelabs/steerling-8b-instruct", trust_remote_code=True)
generator = SteerlingGenerator.from_model(model, tokenizer, device="cuda")

## 2. Choose a concept and a neutral prompt

Set the target concept and a **neutral** prompt, one that does not mention the concept. Amplify should weave the concept into the output that would otherwise be generic.

Set `CONCEPT_ID`, `CONCEPT_LABEL`, and `CONCEPT_DESCRIPTION` to a concept from your concept table.

In [ ]:
# --- target concept (replace with one from your concept table) ---
CONCEPT_ID = 20353
CONCEPT_LABEL = "Books and publishing"

# neutral chat turn (no mention of the concept)
messages = [
    {"role": "system", "content": "You are a helpful AI assistant. Be concise."},
    {"role": "user", "content": "Tell me something interesting."},
]
PROMPT = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

MAX_NEW_TOKENS = 128
STEPS = 128
# steering strength: reduce if you see repetition, increase if you do not see the concept
ALPHA = 8.0
CUTOFF_TOKENS = 32
INJECT_LAYER = 8

EOT_ID = tokenizer.convert_tokens_to_ids("<|eot_id|>")
print(PROMPT)

## 3. Reference generation

First generate with no intervention. This is the baseline the steered run is compared against. Reference sampling uses `temperature=1.2`, `top_p=0.8`, `repetition_penalty=1.1`.

In [ ]:
ref_config = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS, steps=STEPS,
    temperature=1.2, top_p=0.8, repetition_penalty=1.1, seed=0,
    stop_tokens=[EOT_ID],
)

reference = generator.generate_full(PROMPT, ref_config)
print(reference.text)

## 4. Steer toward the concept (`injection`)

`SteeringConfig.injection` builds the steering config: positive injection of the concept direction on the hard_cutoff schedule. We pass it to `generate_steered`. Steered sampling uses `temperature=0.9`, `top_p=0.7`, `repetition_penalty=1.5`.

In [ ]:
steered_config = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS, steps=STEPS,
    temperature=0.9, top_p=0.7, repetition_penalty=1.5, seed=0,
    stop_tokens=[EOT_ID],
)

steer = SteeringConfig.injection(
    concept_ids=CONCEPT_ID,
    mai_lm_target=ALPHA,
    inject_layer=INJECT_LAYER,
    cutoff_tokens=CUTOFF_TOKENS,
)

steered = generator.generate_steered(PROMPT, steered_config, steer)
print(steered.text)